In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#count = 0
for dirname, _, filenames in os.walk('/kaggle/input/notebooks/t1f3001843/dsai-preprocessing'):
    print(dirname)
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/t1f3001843/dsai-preprocessing
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/__results___files
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/chexpert
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/chexphoto_external
/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/nih


In [2]:
from pathlib import Path
import gc
import json
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# -----------------------
# Experiment configuration
# -----------------------
SEED = 42
BACKBONES = ["densenet121", "resnet50", "efficientnet_b0"]   # shorten this list to save GPU budget
IMAGE_SIZE = 224
NUM_WORKERS = 2
USE_IMAGENET_WEIGHTS = True
EVAL_BATCH_SIZE = 64        # eval has no gradients, so a bigger batch is faster and does not change results
MIN_POS = 10                # min positives AND negatives needed to report a label's AUROC

LABELS = [
    "Pleural Effusion",
    "Cardiomegaly",
    "Pneumothorax",
    "Atelectasis",
    "Consolidation",
    "Lung Lesion",
]

# Patient-level split: 1/TEST_SPLITS of patients -> test, then 1/TUNE_SPLITS of the rest -> tuning
TEST_SPLITS = 10            # ~10% of the pool
TUNE_SPLITS = 9             # ~10% of the pool (1/9 of the remaining 90%)

# Stage 1: light-sample search
SEARCH_TRAIN_SIZE = 10_000
SEARCH_VAL_SIZE = 4_000
N_TRIALS = 8
TUNING_EPOCHS = 4

# Stage 2: final retrain of the winning config
FINAL_TRAIN_SIZE = 40_000
FINAL_VAL_SIZE = 8_000
FINAL_EPOCHS = 5

STUDY_TAG = "v1"            # change to start a fresh Optuna study

PREPROCESSED_ROOT = Path("/kaggle/input/notebooks/t1f3001843/dsai-preprocessing")
OUTPUT_DIR = Path("/kaggle/working/multi_backbone_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

Device: cuda
GPU: Tesla T4


In [3]:
#!pip install torchvision optuna

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------- settings ----------------
EXTRA_VALID_PATIENTS = 6000   # patients moved from train to valid (about 3 images each, so ~9k images)
VALID_SEED = 42
# ------------------------------------------

manifest_path = PREPROCESSED_ROOT / "outputs/processed/chexpert_manifest.csv"
if not manifest_path.exists():
    candidates = list(PREPROCESSED_ROOT.rglob("*chexpert*manifest*.csv"))
    if not candidates:
        raise FileNotFoundError(f"Could not find a CheXpert manifest under {PREPROCESSED_ROOT}")
    manifest_path = candidates[0]

print(f"Manifest: {manifest_path}")
manifest = pd.read_csv(manifest_path)

required_columns = {"split", "patient_id", "shard_index", "shard_path", *LABELS}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Manifest is missing columns: {sorted(missing_columns)}")

# One-time index of .npy shard files by filename (avoids searching for every row)
shard_files_by_name = {}
for path in PREPROCESSED_ROOT.rglob("*.npy"):
    shard_files_by_name.setdefault(path.name, []).append(path)


def resolve_shard_path(value):
    candidate = Path(str(value))
    for path in [candidate, PREPROCESSED_ROOT / candidate, manifest_path.parent / candidate]:
        if path.is_file():
            return str(path)
    matches = shard_files_by_name.get(candidate.name, [])
    chexpert_matches = [p for p in matches if "chexpert" in str(p).lower()]
    if len(chexpert_matches) == 1:
        return str(chexpert_matches[0])
    if len(matches) == 1:
        return str(matches[0])
    if len(chexpert_matches) > 1:
        raise ValueError(f"Ambiguous CheXpert shard filename: {candidate.name}")
    return None


manifest["shard_path"] = manifest["shard_path"].map(resolve_shard_path)
missing_shards = manifest["shard_path"].isna().sum()
if missing_shards:
    raise FileNotFoundError(f"Could not resolve {missing_shards:,} manifest rows to .npy shards.")

manifest["shard_index"] = pd.to_numeric(manifest["shard_index"], errors="coerce")
if manifest["shard_index"].isna().any():
    raise ValueError("Some manifest rows have a missing/non-numeric shard_index.")
manifest["shard_index"] = manifest["shard_index"].astype(int)
manifest["split"] = manifest["split"].astype(str).str.lower()
manifest["patient_id"] = manifest["patient_id"].astype(str)

train_all = manifest[manifest["split"].eq("train")].copy()
valid_all = manifest[manifest["split"].isin(["valid", "validation"])].copy()
if train_all.empty or valid_all.empty:
    raise ValueError(f"Expected non-empty train and valid splits; got {len(train_all)} and {len(valid_all)}.")

# ---- enlarge valid: move whole patients from train into valid ----
valid_all["valid_source"] = "official"

rng = np.random.default_rng(VALID_SEED)
train_patients = np.array(sorted(train_all["patient_id"].unique()))
if EXTRA_VALID_PATIENTS >= len(train_patients):
    raise ValueError("EXTRA_VALID_PATIENTS must be smaller than the number of train patients.")
extra_patients = set(rng.choice(train_patients, size=EXTRA_VALID_PATIENTS, replace=False))

extra_mask = train_all["patient_id"].isin(extra_patients)
extra_valid = train_all[extra_mask].copy()
extra_valid["valid_source"] = "carved_from_train"

train_all = train_all[~extra_mask].copy()
valid_all = pd.concat([valid_all, extra_valid], ignore_index=True)

# Safety check: no patient appears in both train and valid
overlap = set(train_all["patient_id"]) & set(valid_all["patient_id"])
if overlap:
    raise ValueError(f"{len(overlap)} patients appear in both train and valid.")

print(manifest["split"].value_counts())
print(f"train_all: {len(train_all):,} images / {train_all['patient_id'].nunique():,} patients")
print(f"valid_all: {len(valid_all):,} images / {valid_all['patient_id'].nunique():,} patients")
print(valid_all["valid_source"].value_counts())

Manifest: /kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/chexpert_manifest.csv
split
train    191027
valid       202
Name: count, dtype: int64
train_all: 172,924 images / 58,534 patients
valid_all: 18,305 images / 6,200 patients
valid_source
carved_from_train    18103
official               202
Name: count, dtype: int64


In [5]:
RARE_FIRST = ["Lung Lesion", "Pneumothorax", "Consolidation",
              "Atelectasis", "Cardiomegaly", "Pleural Effusion"]


def strata(df):
    """One stratum per row: the rarest positive label it has, or 'none'."""
    pos = np.column_stack(
        [pd.to_numeric(df[c], errors="coerce").eq(1).to_numpy() for c in RARE_FIRST]
    )
    return np.where(pos.any(axis=1), pos.argmax(axis=1), len(RARE_FIRST))


def patient_holdout(df, n_splits):
    """Hold out ~1/n_splits of PATIENTS, stratified on rare labels. Returns (kept, held_out)."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    keep, hold = next(sgkf.split(df, strata(df), groups=df["patient_id"]))
    return df.iloc[keep].reset_index(drop=True), df.iloc[hold].reset_index(drop=True)


rest, val_final = patient_holdout(train_all, n_splits=TEST_SPLITS)
train_data, val_tune = patient_holdout(rest, n_splits=TUNE_SPLITS)
val_official = valid_all.reset_index(drop=True)

split_sets = {"train": train_data, "tune": val_tune, "test": val_final, "official": val_official}
names = list(split_sets)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        shared = set(split_sets[a]["patient_id"]) & set(split_sets[b]["patient_id"])
        assert not shared, f"Patient overlap between {a} and {b}: {len(shared)} patients"

total = len(train_all)
for name, d in split_sets.items():
    positives = {c: int(pd.to_numeric(d[c], errors="coerce").eq(1).sum()) for c in RARE_FIRST}
    share = f"{len(d) / total:.1%}" if name != "official" else "-"
    print(f"{name:9s} n={len(d):>8,} ({share:>6s} of train pool)  patients={d['patient_id'].nunique():>7,}  positives={positives}")

train     n= 138,361 ( 80.0% of train pool)  patients= 46,814  positives={'Lung Lesion': 5109, 'Pneumothorax': 12848, 'Consolidation': 9417, 'Atelectasis': 21505, 'Cardiomegaly': 16810, 'Pleural Effusion': 55482}
tune      n=  16,883 (  9.8% of train pool)  patients=  5,863  positives={'Lung Lesion': 640, 'Pneumothorax': 1389, 'Consolidation': 1091, 'Atelectasis': 2624, 'Cardiomegaly': 2044, 'Pleural Effusion': 6855}
test      n=  17,680 ( 10.2% of train pool)  patients=  5,857  positives={'Lung Lesion': 639, 'Pneumothorax': 1712, 'Consolidation': 1268, 'Atelectasis': 2699, 'Cardiomegaly': 2290, 'Pleural Effusion': 7264}
official  n=  18,305 (     - of train pool)  patients=  6,200  positives={'Lung Lesion': 653, 'Pneumothorax': 1751, 'Consolidation': 1239, 'Atelectasis': 2967, 'Cardiomegaly': 2307, 'Pleural Effusion': 7362}


In [6]:
def label_diagnostics(df, name):
    rows = []
    for c in LABELS:
        v = pd.to_numeric(df[c], errors="coerce")
        pos, neg, unc, blank = int((v == 1).sum()), int((v == 0).sum()), int((v == -1).sum()), int(v.isna().sum())
        rows.append({"label": c, "positive": pos, "negative": neg, "uncertain(-1)": unc,
                     "blank": blank, "positive_rate": f"{pos / len(df):.2%}"})
    print(f"\n{name} (n={len(df):,})")
    display(pd.DataFrame(rows).set_index("label"))


for name, d in split_sets.items():
    label_diagnostics(d, name)


train (n=138,361)


,positive,negative,uncertain(-1),blank,positive_rate
label,,,,,
Pleural Effusion,55482,18266,6961,57652,40.10%
Cardiomegaly,16810,5738,4901,110912,12.15%
Pneumothorax,12848,34644,1992,88877,9.29%
Atelectasis,21505,735,21566,94555,15.54%
Consolidation,9417,14064,17680,97200,6.81%
Lung Lesion,5109,563,803,131886,3.69%



tune (n=16,883)


,positive,negative,uncertain(-1),blank,positive_rate
label,,,,,
Pleural Effusion,6855,2328,820,6880,40.60%
Cardiomegaly,2044,696,587,13556,12.11%
Pneumothorax,1389,4314,216,10964,8.23%
Atelectasis,2624,107,2590,11562,15.54%
Consolidation,1091,1783,2102,11907,6.46%
Lung Lesion,640,96,105,16042,3.79%



test (n=17,680)


,positive,negative,uncertain(-1),blank,positive_rate
label,,,,,
Pleural Effusion,7264,2296,867,7253,41.09%
Cardiomegaly,2290,716,584,14090,12.95%
Pneumothorax,1712,4364,236,11368,9.68%
Atelectasis,2699,92,2892,11997,15.27%
Consolidation,1268,1857,2291,12264,7.17%
Lung Lesion,639,51,120,16870,3.61%



official (n=18,305)


,positive,negative,uncertain(-1),blank,positive_rate
label,,,,,
Pleural Effusion,7362,2547,930,7466,40.22%
Cardiomegaly,2307,855,635,14508,12.60%
Pneumothorax,1751,4696,264,11594,9.57%
Atelectasis,2967,218,2815,12305,16.21%
Consolidation,1239,2074,2308,12684,6.77%
Lung Lesion,653,267,80,17305,3.57%


In [7]:
def apply_label_policy(df):
    """Blank -> 0 (negative). Uncertain (-1) -> target 0 AND mask 0 (ignored in loss and eval)."""
    if f"{LABELS[0]}__mask" in df.columns:
        raise RuntimeError("Label policy already applied to this frame. Re-run the split cell (cell 3) first.")
    df = df.copy()
    for c in LABELS:
        raw = pd.to_numeric(df[c], errors="coerce")
        df[f"{c}__mask"] = (raw != -1).astype(np.float32)   # NaN != -1 -> True: blanks stay as negatives
        df[c] = raw.eq(1).astype(np.float32)
    return df


train_data = apply_label_policy(train_data)
val_tune = apply_label_policy(val_tune)
val_final = apply_label_policy(val_final)
val_official = apply_label_policy(val_official)


def sample_frame(df, n):
    return df.sample(n=min(n, len(df)), random_state=SEED).reset_index(drop=True)


search_train = sample_frame(train_data, SEARCH_TRAIN_SIZE)
search_val = sample_frame(val_tune, SEARCH_VAL_SIZE)
final_train = sample_frame(train_data, FINAL_TRAIN_SIZE)
final_val = sample_frame(val_tune, FINAL_VAL_SIZE)

for name, d in [("search_train", search_train), ("search_val", search_val),
                ("final_train", final_train), ("final_val", final_val)]:
    pos = {c: int(d[c].sum()) for c in LABELS}
    masked = {c: int((d[f"{c}__mask"] == 0).sum()) for c in LABELS}
    print(f"{name:13s} n={len(d):>7,}  positives={pos}")
    print(f"{'':13s} uncertain (masked)={masked}")

search_train  n= 10,000  positives={'Pleural Effusion': 3911, 'Cardiomegaly': 1206, 'Pneumothorax': 964, 'Atelectasis': 1536, 'Consolidation': 654, 'Lung Lesion': 361}
              uncertain (masked)={'Pleural Effusion': 531, 'Cardiomegaly': 364, 'Pneumothorax': 161, 'Atelectasis': 1564, 'Consolidation': 1336, 'Lung Lesion': 73}
search_val    n=  4,000  positives={'Pleural Effusion': 1602, 'Cardiomegaly': 478, 'Pneumothorax': 340, 'Atelectasis': 601, 'Consolidation': 257, 'Lung Lesion': 145}
              uncertain (masked)={'Pleural Effusion': 174, 'Cardiomegaly': 144, 'Pneumothorax': 52, 'Atelectasis': 602, 'Consolidation': 491, 'Lung Lesion': 34}
final_train   n= 40,000  positives={'Pleural Effusion': 15922, 'Cardiomegaly': 4892, 'Pneumothorax': 3733, 'Atelectasis': 6234, 'Consolidation': 2724, 'Lung Lesion': 1485}
              uncertain (masked)={'Pleural Effusion': 2030, 'Cardiomegaly': 1456, 'Pneumothorax': 595, 'Atelectasis': 6263, 'Consolidation': 5152, 'Lung Lesion': 262}
fi

In [8]:
def n_steps(n, batch_size):
    return math.ceil(n / batch_size)


stages = [
    ("Search (per trial)", TUNING_EPOCHS, len(search_train), len(search_val)),
    ("Final retrain", FINAL_EPOCHS, len(final_train), len(final_val)),
]

rows = []
for stage, epochs, n_train, n_val in stages:
    for epoch in range(1, epochs + 1):
        rows.append({
            "stage": stage, "epoch": epoch,
            "train_images": n_train,
            "steps_bs16": n_steps(n_train, 16), "steps_bs32": n_steps(n_train, 32),
            "val_images": n_val, "val_batches": n_steps(n_val, EVAL_BATCH_SIZE),
            "cum_train_images": n_train * epoch, "cum_val_images": n_val * epoch,
        })
epoch_table = pd.DataFrame(rows)
display(epoch_table)

search_train_passes = N_TRIALS * TUNING_EPOCHS * len(search_train)
search_val_passes = N_TRIALS * TUNING_EPOCHS * len(search_val)
final_train_passes = FINAL_EPOCHS * len(final_train)
final_val_passes = FINAL_EPOCHS * len(final_val)
test_passes = len(val_final) + len(val_official)
n_back = len(BACKBONES)

print(f"\nWorst case per backbone ({N_TRIALS} trials x {TUNING_EPOCHS} epochs search, then {FINAL_EPOCHS}-epoch final):")
print(f"  search : train passes {search_train_passes:>10,} | validation passes {search_val_passes:>9,}")
print(f"  final  : train passes {final_train_passes:>10,} | validation passes {final_val_passes:>9,}")
print(f"  test   : one-off passes {test_passes:,} (val_final + val_official)")
print(f"  total  : train passes {search_train_passes + final_train_passes:>10,} | "
      f"eval passes {search_val_passes + final_val_passes + test_passes:>9,}")
print(f"\nAll {n_back} backbone(s): train passes {n_back * (search_train_passes + final_train_passes):,} | "
      f"eval passes {n_back * (search_val_passes + final_val_passes + test_passes):,}")
orig_train = 12 * 8 * 17_508 + 8 * 17_508
print(f"\nOriginal notebook, per backbone (for reference): train passes ~{orig_train:,} "
      f"-> new plan is {orig_train / (search_train_passes + final_train_passes):.1f}x fewer training passes in the worst case.")

,stage,epoch,train_images,steps_bs16,steps_bs32,val_images,val_batches,cum_train_images,cum_val_images
0,Search (per trial),1,10000,625,313,4000,63,10000,4000
1,Search (per trial),2,10000,625,313,4000,63,20000,8000
2,Search (per trial),3,10000,625,313,4000,63,30000,12000
3,Search (per trial),4,10000,625,313,4000,63,40000,16000
4,Final retrain,1,40000,2500,1250,8000,125,40000,8000
5,Final retrain,2,40000,2500,1250,8000,125,80000,16000
6,Final retrain,3,40000,2500,1250,8000,125,120000,24000
7,Final retrain,4,40000,2500,1250,8000,125,160000,32000
8,Final retrain,5,40000,2500,1250,8000,125,200000,40000



Worst case per backbone (8 trials x 4 epochs search, then 5-epoch final):
  search : train passes    320,000 | validation passes   128,000
  final  : train passes    200,000 | validation passes    40,000
  test   : one-off passes 35,985 (val_final + val_official)
  total  : train passes    520,000 | eval passes   203,985

All 3 backbone(s): train passes 1,560,000 | eval passes 611,955

Original notebook, per backbone (for reference): train passes ~1,820,832 -> new plan is 3.5x fewer training passes in the worst case.


In [9]:
class CheXpertShardDataset(Dataset):
    MEAN, STD = 0.449, 0.226   # channel-averaged ImageNet statistics

    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.paths = self.frame["shard_path"].astype(str).to_numpy()
        self.positions = self.frame["shard_index"].astype(int).to_numpy()
        self.targets = self.frame[LABELS].to_numpy(dtype=np.float32)
        self.masks = self.frame[[f"{c}__mask" for c in LABELS]].to_numpy(dtype=np.float32)
        self.shard_cache = {}   # one cache per DataLoader worker
        self.augmentation = transforms.Compose([
            transforms.RandomAffine(degrees=5, translate=(0.02, 0.02)),
            transforms.ColorJitter(contrast=0.05),
        ]) if training else None

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path, pos = self.paths[index], int(self.positions[index])
        if path not in self.shard_cache:
            self.shard_cache[path] = np.load(path, mmap_mode="r", allow_pickle=False)
        shard = self.shard_cache[path]
        if not 0 <= pos < len(shard):
            raise IndexError(f"shard_index {pos} is outside shard of length {len(shard)}: {path}")

        image = np.asarray(shard[pos])
        if image.ndim != 2:
            raise ValueError(f"Expected grayscale (H, W) image; got {image.shape}")
        image = image.astype(np.float32)
        if shard.dtype == np.uint8:        # decide by dtype, not by this image's max
            image /= 255.0

        image = torch.from_numpy(image.copy()).unsqueeze(0)
        if self.augmentation is not None:
            image = self.augmentation(image)
        image = (image - self.MEAN) / self.STD

        return image, torch.from_numpy(self.targets[index]), torch.from_numpy(self.masks[index])


def make_loader(frame, batch_size, training):
    return DataLoader(
        CheXpertShardDataset(frame, training=training),
        batch_size=batch_size,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )


# Sanity check on one sample
ex_image, ex_targets, ex_mask = CheXpertShardDataset(search_train)[0]
print("Image:", tuple(ex_image.shape), ex_image.dtype)
print("Normalised pixel range:", round(float(ex_image.min()), 3), "to", round(float(ex_image.max()), 3))
print("Targets:", ex_targets.tolist())
print("Mask   :", ex_mask.tolist())
if tuple(ex_image.shape[-2:]) != (IMAGE_SIZE, IMAGE_SIZE):
    print(f"WARNING: shard images are {tuple(ex_image.shape[-2:])}, not {IMAGE_SIZE}x{IMAGE_SIZE}. "
          "Models will still run, but update IMAGE_SIZE or the preprocessing if this is unintended.")

Image: (1, 224, 224) torch.float32
Normalised pixel range: -1.987 to 2.438
Targets: [0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
Mask   : [1.0, 1.0, 1.0, 0.0, 0.0, 1.0]


In [10]:
# Inspect ten training samples: how many of the labels are "observed" (not masked)?
inspect_ds = CheXpertShardDataset(search_train)
for i in range(10):
    image, targets, mask = inspect_ds[i]
    print(f"Image {i}: targets={targets.tolist()}  observed={int(mask.sum().item())}/{len(LABELS)}")

Image 0: targets=[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]  observed=4/6
Image 1: targets=[0.0, 0.0, 0.0, 0.0, 0.0, 1.0]  observed=5/6
Image 2: targets=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=6/6
Image 3: targets=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=4/6
Image 4: targets=[0.0, 0.0, 0.0, 0.0, 0.0, 1.0]  observed=6/6
Image 5: targets=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=6/6
Image 6: targets=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=3/6
Image 7: targets=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=5/6
Image 8: targets=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=6/6
Image 9: targets=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]  observed=4/6


In [11]:
def adapt_first_conv_to_grayscale(old_conv):
    """Preserve pretrained RGB filters by averaging their input channels."""
    new_conv = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, dilation=old_conv.dilation,
        groups=old_conv.groups, bias=old_conv.bias is not None,
    )
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias)
    return new_conv


def build_model(backbone, dropout):
    n_out = len(LABELS)
    if backbone == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.densenet121(weights=weights)
        model.features.conv0 = adapt_first_conv_to_grayscale(model.features.conv0)
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier.in_features, n_out))
    elif backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.resnet50(weights=weights)
        model.conv1 = adapt_first_conv_to_grayscale(model.conv1)
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, n_out))
    elif backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.efficientnet_b0(weights=weights)
        model.features[0][0] = adapt_first_conv_to_grayscale(model.features[0][0])
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier[-1].in_features, n_out))
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return model


def masked_bce_loss(logits, targets, mask):
    element_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    return (element_loss * mask).sum() / mask.sum().clamp_min(1.0)


def per_label_auroc(probs, targets, masks):
    """AUROC per label (array, NaN where a label has < MIN_POS positives or negatives among observed rows)."""
    out = np.full(len(LABELS), np.nan)
    for j in range(len(LABELS)):
        keep = masks[:, j].astype(bool)
        y = targets[keep, j]
        if (y == 1).sum() < MIN_POS or (y == 0).sum() < MIN_POS:
            continue
        out[j] = roc_auc_score(y, probs[keep, j])
    return out


def calculate_auroc(logits, targets, masks):
    probs = torch.sigmoid(logits).numpy()
    aucs = per_label_auroc(probs, targets.numpy(), masks.numpy())
    scores = {label: float(a) for label, a in zip(LABELS, aucs)}
    finite = aucs[np.isfinite(aucs)]
    macro = float(finite.mean()) if finite.size else float("nan")
    return macro, scores


def evaluate(model, loader, return_raw=False):
    model.eval()
    all_logits, all_targets, all_masks = [], [], []
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        for images, targets, masks in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            all_logits.append(logits.float().cpu())
            all_targets.append(targets)
            all_masks.append(masks)
    logits, targets, masks = torch.cat(all_logits), torch.cat(all_targets), torch.cat(all_masks)
    macro, scores = calculate_auroc(logits, targets, masks)
    return (macro, scores, (logits, targets, masks)) if return_raw else (macro, scores)


def bootstrap_ci(raw, patient_ids, n_boot=200, seed=SEED):
    """95% CI for macro and per-label AUROC, resampling PATIENTS with replacement."""
    logits, targets, masks = raw
    probs, targets, masks = torch.sigmoid(logits).numpy(), targets.numpy(), masks.numpy()
    codes, uniques = pd.factorize(pd.Series(np.asarray(patient_ids)))
    order = np.argsort(codes, kind="stable")
    bounds = np.searchsorted(codes[order], np.arange(len(uniques) + 1))
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniques))]

    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in pick])
        draws.append(per_label_auroc(probs[idx], targets[idx], masks[idx]))
    draws = np.array(draws)

    def interval(column):
        column = column[np.isfinite(column)]
        return (float("nan"), float("nan")) if column.size < 10 else tuple(np.percentile(column, [2.5, 97.5]))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        macro_draws = np.nanmean(draws, axis=1)
    result = {"macro": interval(macro_draws)}
    for j, label in enumerate(LABELS):
        result[label] = interval(draws[:, j])
    return result


print("Models, masked BCE, AUROC and bootstrap helpers are ready.")

Models, masked BCE, AUROC and bootstrap helpers are ready.


In [12]:
def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_model(backbone, config, epochs, train_frame, tune_frame,
                trial=None, save_path=None, patience=2):
    set_seed(SEED)   # same seed for every trial and backbone
    train_loader = make_loader(train_frame, config["batch_size"], training=True)
    tune_loader = make_loader(tune_frame, EVAL_BATCH_SIZE, training=False)
    model = build_model(backbone, config["dropout"]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=config["learning_rate"] * 0.05
    )
    scaler = make_scaler()

    best_auroc, best_scores, best_epoch = -float("inf"), None, 0
    epochs_without_improvement = 0
    history = []
    pruned = False

    try:
        for epoch in range(epochs):
            model.train()
            start = time.perf_counter()
            running_loss, batches = 0.0, 0
            for images, targets, masks in train_loader:
                images = images.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits = model(images)
                loss = masked_bce_loss(logits.float(), targets, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.detach().cpu())
                batches += 1
            scheduler.step()

            val_auroc, per_finding = evaluate(model, tune_loader)
            if not np.isfinite(val_auroc):
                raise RuntimeError(
                    "Tuning macro AUROC is NaN: no label has >= MIN_POS positives and negatives "
                    "in the tuning frame. Increase the validation sample size or lower MIN_POS."
                )
            elapsed = time.perf_counter() - start
            history.append({
                "epoch": epoch + 1, "train_images": len(train_frame), "val_images": len(tune_frame),
                "train_loss": running_loss / max(batches, 1), "val_macro_auroc": val_auroc,
                "lr": optimizer.param_groups[0]["lr"], "seconds": elapsed,
            })
            print(f"{backbone} | epoch {epoch + 1}/{epochs} | train n={len(train_frame):,} val n={len(tune_frame):,} | "
                  f"loss={history[-1]['train_loss']:.4f} | val_macro_AUROC={val_auroc:.4f} | {elapsed:.0f}s")

            if val_auroc > best_auroc:
                best_auroc, best_scores, best_epoch = val_auroc, per_finding, epoch + 1
                epochs_without_improvement = 0
                if save_path is not None:
                    torch.save({"model_state_dict": model.state_dict(), "config": config,
                                "epoch": best_epoch, "backbone": backbone,
                                "tuning_macro_auroc": best_auroc}, save_path)
            else:
                epochs_without_improvement += 1

            if trial is not None:
                trial.report(val_auroc, epoch)
                if trial.should_prune():
                    pruned = True
                    break
            if epochs_without_improvement >= patience:
                print("Early stopping: tuning AUROC did not improve.")
                break
    finally:
        del model, optimizer, scheduler, scaler, train_loader, tune_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if pruned:
        raise optuna.TrialPruned()
    return {"macro_auroc": best_auroc, "per_finding_auroc": best_scores,
            "best_epoch": best_epoch, "config": config, "history": history}


print("train_model is ready.")

train_model is ready.


In [13]:
def make_objective(backbone):
    def objective(trial):
        config = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32]),
            "dropout": trial.suggest_categorical("dropout", [0.0, 0.2, 0.4]),
            "weight_decay": trial.suggest_float("weight_decay", 1e-3, 1e-1, log=True),
        }
        result = train_model(
            backbone, config, epochs=TUNING_EPOCHS, trial=trial,
            train_frame=search_train, tune_frame=search_val,
        )
        trial.set_user_attr("best_epoch", result["best_epoch"])
        trial.set_user_attr("per_finding_auroc", result["per_finding_auroc"])
        return result["macro_auroc"]
    return objective


studies, best_configs = {}, {}
for backbone in BACKBONES:
    print(f"\n{'=' * 65}\nTuning {backbone}: {N_TRIALS} trials, up to {TUNING_EPOCHS} epochs each\n{'=' * 65}")
    backbone_dir = OUTPUT_DIR / backbone
    backbone_dir.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        study_name=f"chexpert_{backbone}_{STUDY_TAG}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
        storage=f"sqlite:///{backbone_dir}/study_{STUDY_TAG}.db",
        load_if_exists=True,
    )
    remaining = max(0, N_TRIALS - len(study.trials))
    print(f"{len(study.trials)} trial(s) already stored; running {remaining} more.")
    study.optimize(make_objective(backbone), n_trials=remaining)
    studies[backbone] = study

    study.trials_dataframe().to_csv(backbone_dir / "optuna_trials.csv", index=False)
    best_configs[backbone] = dict(study.best_trial.params)
    (backbone_dir / "best_config.json").write_text(json.dumps(best_configs[backbone], indent=2))
    print(f"{backbone}: best search macro AUROC = {study.best_value:.4f}")
    print(json.dumps(best_configs[backbone], indent=2))

print("\nSearch complete for all backbones.")


Tuning densenet121: 8 trials, up to 4 epochs each


[I 2026-11-17 16:20:17,554] A new study created in RDB with name: chexpert_densenet121_v1


0 trial(s) already stored; running 8 more.
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 235MB/s]


densenet121 | epoch 1/4 | train n=10,000 val n=4,000 | loss=0.3727 | val_macro_AUROC=0.7211 | 429s
densenet121 | epoch 2/4 | train n=10,000 val n=4,000 | loss=0.3037 | val_macro_AUROC=0.7471 | 65s
densenet121 | epoch 3/4 | train n=10,000 val n=4,000 | loss=0.2798 | val_macro_AUROC=0.7564 | 65s
densenet121 | epoch 4/4 | train n=10,000 val n=4,000 | loss=0.2584 | val_macro_AUROC=0.7575 | 65s


[I 2026-11-17 16:30:42,952] Trial 0 finished with value: 0.7574932894527865 and parameters: {'learning_rate': 4.3284502212938785e-05, 'batch_size': 16, 'dropout': 0.0, 'weight_decay': 0.0013066739238053278}. Best is trial 0 with value: 0.7574932894527865.


... (47 outputs left out)

[I 2026-11-17 17:32:16,407] Trial 6 pruned. 


efficientnet_b0 | epoch 1/4 | train n=10,000 val n=4,000 | loss=0.5307 | val_macro_AUROC=0.5709 | 32s
efficientnet_b0 | epoch 2/4 | train n=10,000 val n=4,000 | loss=0.3786 | val_macro_AUROC=0.6143 | 31s


[I 2026-11-17 17:33:19,689] Trial 7 pruned. 


efficientnet_b0: best search macro AUROC = 0.7488
{
  "learning_rate": 0.00031332776986527485,
  "batch_size": 32,
  "dropout": 0.2,
  "weight_decay": 0.0015815209909357238
}

Search complete for all backbones.


In [14]:
if "best_configs" not in globals() or not best_configs:
    best_configs = {b: json.loads((OUTPUT_DIR / b / "best_config.json").read_text()) for b in BACKBONES}

summary_rows, per_label_rows, history_rows = [], [], []

for backbone in BACKBONES:
    print(f"\n{'=' * 65}\nFinal evaluation: {backbone}\n{'=' * 65}")
    backbone_dir = OUTPUT_DIR / backbone
    checkpoint_path = backbone_dir / "best_model.pt"
    config = best_configs[backbone]

    # 1) retrain best config on more data; best epoch is chosen on final_val
    retrain = train_model(
        backbone, config, epochs=FINAL_EPOCHS,
        train_frame=final_train, tune_frame=final_val, save_path=checkpoint_path,
    )
    for record in retrain["history"]:
        history_rows.append({"backbone": backbone, **record})

    # 2) reload the best checkpoint
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = build_model(backbone, checkpoint["config"]["dropout"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)

    # 3) test set, scored once, with patient-level bootstrap CIs
    test_loader = make_loader(val_final, EVAL_BATCH_SIZE, training=False)
    test_macro, test_scores, test_raw = evaluate(model, test_loader, return_raw=True)
    test_ci = bootstrap_ci(test_raw, val_final["patient_id"].to_numpy())

    # 4) official 202-image set (second opinion only)
    official_loader = make_loader(val_official, EVAL_BATCH_SIZE, training=False)
    official_macro, official_scores = evaluate(model, official_loader)

    summary_rows.append({
        "backbone": backbone,
        "search_best_macro_auroc": studies[backbone].best_value if "studies" in globals() and backbone in studies else float("nan"),
        "final_val_macro_auroc": retrain["macro_auroc"],
        "test_macro_auroc": test_macro,
        "test_ci_low": test_ci["macro"][0], "test_ci_high": test_ci["macro"][1],
        "official_macro_auroc": official_macro,
        "best_epoch": checkpoint["epoch"],
        **config,
    })
    for label in LABELS:
        per_label_rows.append({
            "backbone": backbone, "label": label,
            "test_auroc": test_scores[label],
            "test_ci_low": test_ci[label][0], "test_ci_high": test_ci[label][1],
            "official_auroc": official_scores[label],
        })
    (backbone_dir / "val_final_results.json").write_text(json.dumps({
        "summary": summary_rows[-1], "test_per_label": test_scores,
        "test_ci": {k: list(v) for k, v in test_ci.items()},
        "official_per_label": official_scores,
    }, indent=2, default=float))

    print(f"{backbone}: test macro AUROC = {test_macro:.4f} "
          f"(95% CI {test_ci['macro'][0]:.4f} to {test_ci['macro'][1]:.4f}) | official = {official_macro:.4f}")
    del model, test_loader, official_loader, checkpoint, test_raw
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
per_label_df = pd.DataFrame(per_label_rows)
history_df = pd.DataFrame(history_rows)
best_row = summary_df.loc[summary_df["final_val_macro_auroc"].idxmax()]
summary_df["selected_by_final_val"] = summary_df["backbone"].eq(best_row["backbone"])

summary_df.to_csv(OUTPUT_DIR / "architecture_comparison.csv", index=False)
per_label_df.to_csv(OUTPUT_DIR / "val_final_per_finding_auroc.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print(f"\nSelected backbone (by final_val macro AUROC, not by test): {best_row['backbone']}")
display(summary_df.sort_values("final_val_macro_auroc", ascending=False))
print("\nPer-label test AUROC with 95% CI, and official-set AUROC:")
display(per_label_df)
print("\nEpoch-wise history of the final retrains (sizes, loss, AUROC, time):")
display(history_df)


Final evaluation: densenet121
densenet121 | epoch 1/5 | train n=40,000 val n=8,000 | loss=0.3213 | val_macro_AUROC=0.7565 | 523s
densenet121 | epoch 2/5 | train n=40,000 val n=8,000 | loss=0.3021 | val_macro_AUROC=0.7657 | 252s
densenet121 | epoch 3/5 | train n=40,000 val n=8,000 | loss=0.2906 | val_macro_AUROC=0.7906 | 253s
densenet121 | epoch 4/5 | train n=40,000 val n=8,000 | loss=0.2743 | val_macro_AUROC=0.7976 | 251s
densenet121 | epoch 5/5 | train n=40,000 val n=8,000 | loss=0.2536 | val_macro_AUROC=0.7975 | 250s
densenet121: test macro AUROC = 0.7913 (95% CI 0.7853 to 0.7971) | official = 0.7935

Final evaluation: resnet50
resnet50 | epoch 1/5 | train n=40,000 val n=8,000 | loss=0.3239 | val_macro_AUROC=0.7655 | 200s
resnet50 | epoch 2/5 | train n=40,000 val n=8,000 | loss=0.2986 | val_macro_AUROC=0.7812 | 199s
resnet50 | epoch 3/5 | train n=40,000 val n=8,000 | loss=0.2822 | val_macro_AUROC=0.7846 | 199s
resnet50 | epoch 4/5 | train n=40,000 val n=8,000 | loss=0.2567 | val_mac

,backbone,search_best_macro_auroc,final_val_macro_auroc,test_macro_auroc,test_ci_low,test_ci_high,official_macro_auroc,best_epoch,learning_rate,batch_size,dropout,weight_decay,selected_by_final_val
0,densenet121,0.763024,0.797561,0.791323,0.785338,0.797097,0.793458,4,0.000188,16,0.0,0.023452,True
2,efficientnet_b0,0.748845,0.791979,0.788726,0.782342,0.794718,0.790569,5,0.000313,32,0.2,0.001582,False
1,resnet50,0.758714,0.786037,0.781587,0.774487,0.787629,0.782116,4,0.000043,16,0.4,0.007403,False



Per-label test AUROC with 95% CI, and official-set AUROC:


,backbone,label,test_auroc,test_ci_low,test_ci_high,official_auroc
0,densenet121,Pleural Effusion,0.879995,0.871853,0.886346,0.882791
1,densenet121,Cardiomegaly,0.867063,0.857544,0.876691,0.865206
2,densenet121,Pneumothorax,0.828288,0.809093,0.841807,0.848077
3,densenet121,Atelectasis,0.697791,0.684755,0.711425,0.703538
4,densenet121,Consolidation,0.732636,0.716585,0.746273,0.725673
5,densenet121,Lung Lesion,0.742165,0.714760,0.766556,0.735462
6,resnet50,Pleural Effusion,0.869395,0.861367,0.876731,0.872551
7,resnet50,Cardiomegaly,0.848285,0.836107,0.859048,0.843392
8,resnet50,Pneumothorax,0.824348,0.806724,0.837076,0.832558
9,resnet50,Atelectasis,0.690853,0.676832,0.702854,0.691879



Epoch-wise history of the final retrains (sizes, loss, AUROC, time):


,backbone,epoch,train_images,val_images,train_loss,val_macro_auroc,lr,seconds
0,densenet121,1,40000,8000,0.321299,0.756502,0.000171,522.750906
1,densenet121,2,40000,8000,0.302106,0.765734,0.000127,251.974031
2,densenet121,3,40000,8000,0.290642,0.790588,0.000071,253.222544
3,densenet121,4,40000,8000,0.274276,0.797561,0.000027,250.977369
4,densenet121,5,40000,8000,0.253600,0.797471,0.000009,249.762408
5,resnet50,1,40000,8000,0.323890,0.765463,0.000039,199.884316
6,resnet50,2,40000,8000,0.298616,0.781220,0.000029,199.421912
7,resnet50,3,40000,8000,0.282179,0.784616,0.000016,199.005307
8,resnet50,4,40000,8000,0.256695,0.786037,0.000006,199.238714
9,resnet50,5,40000,8000,0.226639,0.778359,0.000002,198.889623


In [15]:
for backbone in BACKBONES:
    t = pd.read_csv(OUTPUT_DIR / backbone / "optuna_trials.csv")
    cols = [c for c in ["number", "state", "value", "params_learning_rate", "params_batch_size",
                        "params_dropout", "params_weight_decay", "user_attrs_best_epoch"] if c in t.columns]
    t = t[cols].sort_values("value", ascending=False)
    done = t.loc[t["state"] == "COMPLETE", "value"]
    print(f"\n{backbone}: {len(t)} trials, {int((t['state'] == 'PRUNED').sum())} pruned | "
          f"best {done.max():.4f} | median {done.median():.4f} | gap {done.max() - done.median():.4f}")
    display(t)


densenet121: 8 trials, 3 pruned | best 0.7630 | median 0.7575 | gap 0.0055


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
6,6,COMPLETE,0.763024,0.000188,16,0.0,0.023452,4.0
5,5,COMPLETE,0.760019,0.000090,16,0.0,0.051768,3.0
0,0,COMPLETE,0.757493,0.000043,16,0.0,0.001307,4.0
1,1,COMPLETE,0.757347,0.000296,32,0.2,0.002659,4.0
4,4,PRUNED,0.754479,0.000034,16,0.0,0.001359,NaN
7,7,PRUNED,0.732740,0.000401,16,0.4,0.083323,NaN
2,2,COMPLETE,0.726731,0.000020,32,0.0,0.016738,4.0
3,3,PRUNED,0.686617,0.000017,32,0.2,0.010677,NaN



resnet50: 8 trials, 2 pruned | best 0.7587 | median 0.7507 | gap 0.0080


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
6,6,COMPLETE,0.758714,0.000043,16,0.4,0.007403,3.0
4,4,COMPLETE,0.752183,0.000034,16,0.0,0.001359,2.0
0,0,COMPLETE,0.751387,0.000043,16,0.0,0.001307,4.0
5,5,COMPLETE,0.750084,0.000088,16,0.0,0.051744,3.0
2,2,COMPLETE,0.748693,0.000020,32,0.0,0.016738,4.0
1,1,COMPLETE,0.747800,0.000296,32,0.2,0.002659,4.0
7,7,PRUNED,0.746709,0.000069,16,0.4,0.011075,NaN
3,3,PRUNED,0.733248,0.000017,32,0.2,0.010677,NaN



efficientnet_b0: 8 trials, 3 pruned | best 0.7488 | median 0.7442 | gap 0.0046


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
5,5,COMPLETE,0.748845,0.000313,32,0.2,0.001582,4.0
1,1,COMPLETE,0.747791,0.000296,32,0.2,0.002659,4.0
4,4,COMPLETE,0.744228,0.000228,32,0.2,0.051632,4.0
0,0,COMPLETE,0.712873,0.000043,16,0.0,0.001307,4.0
6,6,PRUNED,0.710639,0.000085,32,0.4,0.006380,NaN
2,2,COMPLETE,0.657447,0.000020,32,0.0,0.016738,4.0
3,3,PRUNED,0.621950,0.000017,32,0.2,0.010677,NaN
7,7,PRUNED,0.614280,0.000016,32,0.2,0.001254,NaN


In [16]:
import shutil

# Save the exact split membership so you can verify the test set is identical in later sessions
pd.concat(
    [d[["patient_id", "shard_path", "shard_index"]].assign(set=name)
     for name, d in {"train": train_data, "tune": val_tune,
                     "test": val_final, "official": val_official}.items()]
).to_csv(OUTPUT_DIR / "split_assignment.csv.gz", index=False)

zip_path = shutil.make_archive("/kaggle/working/chexpert_tuning_checkpoint", "zip",
                               root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
n_files = sum(1 for p in OUTPUT_DIR.rglob("*") if p.is_file())
print(f"{zip_path}: {Path(zip_path).stat().st_size / 1e6:.0f} MB, {n_files} files")

/kaggle/working/chexpert_tuning_checkpoint.zip: 130 MB, 19 files
